# 1.09 — Functions in depth — solutions

Lesson: [`lessons/1.09_functions_in_depth.md`](../lessons/1.09_functions_in_depth.md)

> **Reference solutions.** Open only after you have solved the problem yourself or climbed the hint ladder. Compare, don't copy: then re-solve from blank in a few days.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


## Problem 1 — describe any number of values

`describe(*nums)` returns `(count, mean)` with the mean rounded to 2 decimals. No arguments → `(0, 0.0)`.

In [ ]:
def describe(*nums):
    if not nums:
        return (0, 0.0)
    return (len(nums), round(sum(nums) / len(nums), 2))

In [ ]:
check(describe, [((1, 2, 3, 4), (4, 2.5)), ((), (0, 0.0)), ((10,), (1, 10.0))])

## Problem 2 — build a URL

`build_url(url, **params)` → `build_url("https://api.x.com/rates", base="USD", date="2026-09-01")` returns `"https://api.x.com/rates?base=USD&date=2026-09-01"`. No params → just `url`.

*(Why is the first parameter called `url` and not `base`? Try renaming it to `base` after you've solved it and see what breaks.)*

In [ ]:
def build_url(url, **params):
    if not params:
        return url
    query = "&".join(f"{k}={v}" for k, v in params.items())
    return f"{url}?{query}"
# If the first parameter were named `base`, the call build_url(..., base="USD") would pass `base` twice -> TypeError.

In [ ]:
print(build_url("https://api.x.com/rates", base="USD", date="2026-09-01"))
expect("two params", build_url("https://api.x.com/rates", base="USD", date="2026-09-01") == "https://api.x.com/rates?base=USD&date=2026-09-01")
expect("no params", build_url("https://a.b") == "https://a.b")
expect("unpacked dict", build_url("u", **{"k": 1}) == "u?k=1")

## Problem 3 — fix the mutable default

This function has the classic bug. Run the demo, see the problem, then fix it.

In [ ]:
def add_item(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

In [ ]:
first = add_item("tea")
second = add_item("rice")
expect("each call without a basket starts fresh", first == ["tea"] and second == ["rice"])
expect("passing a basket still works", add_item("b", ["a"]) == ["a", "b"])

## Problem 4 — keyword-only options

`apply_discount(price, pct=10, *, round_to=2)` returns the discounted price rounded to `round_to` decimals. `round_to` must be keyword-only.

In [ ]:
def apply_discount(price, pct=10, *, round_to=2):
    return round(price * (1 - pct / 100), round_to)

In [ ]:
check(apply_discount, [((200,), 180.0), ((200, 25), 150.0), ((99.99, 15), 84.99)])
expect("round_to works as keyword", apply_discount(333.333, round_to=0) == 300.0)
try:
    apply_discount(200, 10, 0)
    expect("round_to is keyword-only", False)
except TypeError:
    expect("round_to is keyword-only", True)

## Problem 5 — sort with a key

`sort_by_amount(transactions)` returns a new list sorted by `"amount"` ascending. `sort_by_merchant_then_amount(transactions)` sorts by merchant A→Z, then amount **high→low**.

In [ ]:
def sort_by_amount(transactions):
    return sorted(transactions, key=lambda t: t["amount"])

def sort_by_merchant_then_amount(transactions):
    return sorted(transactions, key=lambda t: (t["merchant"], -t["amount"]))

In [ ]:
T = [{"merchant": "B", "amount": 5}, {"merchant": "A", "amount": 1}, {"merchant": "B", "amount": 9}]
check(sort_by_amount, [((T,), [{"merchant": "A", "amount": 1}, {"merchant": "B", "amount": 5}, {"merchant": "B", "amount": 9}])])
check(sort_by_merchant_then_amount, [((T,), [{"merchant": "A", "amount": 1}, {"merchant": "B", "amount": 9}, {"merchant": "B", "amount": 5}])])

## Problem 6 — your own `map`

`apply_to_all(func, items)` returns a new list with `func` applied to each item.

In [ ]:
def apply_to_all(func, items):
    out = []
    for x in items:
        out.append(func(x))
    return out

In [ ]:
check(apply_to_all, [((abs, [-1, 2, -3]), [1, 2, 3]), ((str.upper, ["a", "b"]), ["A", "B"]), ((len, []), [])])

## Problem 7 — biggest by any measure

`biggest(items, measure)` returns the item for which `measure(item)` is largest (first one on ties). Don't use `max(..., key=...)` — write the loop.

In [ ]:
def biggest(items, measure):
    best = None
    best_score = None
    for x in items:
        score = measure(x)
        if best_score is None or score > best_score:
            best, best_score = x, score
    return best

In [ ]:
check(biggest, [((["kiwi", "banana", "fig"], len), "banana"), (([3, -7, 5], abs), -7)])

## Problem 8 ★ — function factories

`make_converter(rate)` returns a **function** that converts an amount at that rate, rounded to 2 decimals. `compose(f, g)` returns a function that computes `f(g(x))`.

In [ ]:
def make_converter(rate):
    def convert(amount):
        return round(amount * rate, 2)
    return convert

def compose(f, g):
    def composed(x):
        return f(g(x))
    return composed

In [ ]:
usd_to_lkr = make_converter(300.25)
expect("converter works", usd_to_lkr(10) == 3002.5)
add_one = lambda x: x + 1
double = lambda x: x * 2
expect("compose(double, add_one)(3) == 8", compose(double, add_one)(3) == 8)
expect("compose(add_one, double)(3) == 7", compose(add_one, double)(3) == 7)